# 🌱 AgroClima Perú — Microservicio Predictivo

In [55]:
# ── CELDA 1: Instalación de dependencias ──────────────────────
!pip install fastapi uvicorn nest_asyncio requests -q
print('✅ Dependencias instaladas')

✅ Dependencias instaladas


In [56]:
# ── CELDA 2: MICROSERVICIO AGROCLIMÁTICO COMPLETO ─────────────
# Fuente de datos: ENA 2024 (Módulos 1893, 1894, 1895 - INEI/MIDAGRI)
# Clima en tiempo real: Open-Meteo API (sin costo, sin API key)
# ────────────────────────────────────────────────────────────────
import nest_asyncio
import threading
import uvicorn
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
import requests as req_lib
from datetime import datetime

nest_asyncio.apply()

# ══════════════════════════════════════════════════════════════
# BASE DE CONOCIMIENTO - Fuente: ENA 2024 (Módulos 1893/1895)
# ══════════════════════════════════════════════════════════════

REGIONES_DATA = {
    "LAMBAYEQUE": {
        "lat": -6.7011, "lon": -79.9053,
        "cultivos_principales": ["MANGO", "PLATANO", "ARROZ CASCARA", "CAÑA DE AZUCAR", "MAIZ AMARILLO DURO"],
        "cultivos_favorables": {
            # temp_min, temp_max, meses_siembra, meses_cosecha, riesgo_climatico
            "MANGO":            {"t_min": 18, "t_max": 35, "siembra": [11,12,1], "cosecha": [12,1,2], "riesgo": "ESTRES_TERMICO"},
            "PLATANO":          {"t_min": 18, "t_max": 35, "siembra": [1,2,3],   "cosecha": [8,9,10], "riesgo": "EXCESO_LLUVIA"},
            "ARROZ CASCARA":    {"t_min": 20, "t_max": 38, "siembra": [12,1,2],  "cosecha": [5,6,7],  "riesgo": "SEQUIA"},
            "CAÑA DE AZUCAR":   {"t_min": 20, "t_max": 35, "siembra": [1,2,3],   "cosecha": [9,10,11],"riesgo": "SEQUIA"},
            "MAIZ AMARILLO DURO":{"t_min": 18, "t_max": 35, "siembra": [11,12,1],"cosecha": [4,5,6],  "riesgo": "SEQUIA"},
        },
        "perdidas_diego": "Región con pérdidas de productividad del 30-50% en agroexportación por estrés térmico (2024). Afecta principalmente frutas en floración.",
        "alerta_principal": "ESTRES_TERMICO",
        "produccion_ranking": 1,  # Prioridad del ing. Raúl
    },
    "LA LIBERTAD": {
        "lat": -8.1119, "lon": -79.0291,
        "cultivos_principales": ["PALTO", "ARANDANO", "ESPARRAGO", "CAÑA DE AZUCAR", "PLATANO"],
        "cultivos_favorables": {
            "PALTO":     {"t_min": 15, "t_max": 30, "siembra": [3,4,5],   "cosecha": [4,5,6,7], "riesgo": "ESTRES_TERMICO"},
            "ARANDANO":  {"t_min": 10, "t_max": 28, "siembra": [3,4],     "cosecha": [6,7,8],   "riesgo": "ESTRES_TERMICO"},
            "ESPARRAGO": {"t_min": 15, "t_max": 30, "siembra": [4,5],     "cosecha": [9,10,11], "riesgo": "SEQUIA"},
            "CAÑA DE AZUCAR":{"t_min": 18, "t_max": 35,"siembra": [1,2,3],"cosecha": [9,10,11], "riesgo": "SEQUIA"},
            "PLATANO":   {"t_min": 18, "t_max": 35, "siembra": [1,2,3],   "cosecha": [8,9,10],  "riesgo": "EXCESO_LLUVIA"},
        },
        "perdidas_diego": "Caídas de productividad del 30-50% en arándanos y palta por temperaturas inusualmente altas que afectaron la floración (2024).",
        "alerta_principal": "ESTRES_TERMICO",
        "produccion_ranking": 2,
    },
    "ICA": {
        "lat": -14.0678, "lon": -75.7286,
        "cultivos_principales": ["VID", "PALTO", "ESPARRAGO", "PECANO", "MANDARINA"],
        "cultivos_favorables": {
            "VID":       {"t_min": 15, "t_max": 32, "siembra": [8,9],     "cosecha": [12,1,2],  "riesgo": "ESTRES_TERMICO"},
            "PALTO":     {"t_min": 15, "t_max": 30, "siembra": [8,9],     "cosecha": [11,12,1], "riesgo": "SEQUIA"},
            "ESPARRAGO": {"t_min": 15, "t_max": 30, "siembra": [4,5],     "cosecha": [9,10,11], "riesgo": "SEQUIA"},
            "PECANO":    {"t_min": 12, "t_max": 30, "siembra": [8,9],     "cosecha": [10,11,12],"riesgo": "SEQUIA"},
            "MANDARINA": {"t_min": 15, "t_max": 32, "siembra": [8,9],     "cosecha": [5,6,7],   "riesgo": "ESTRES_TERMICO"},
        },
        "perdidas_diego": "Afectada por temperaturas inusualmente altas en agroexportación (uva de mesa y palta). Productividad aún bajo niveles prepandemia.",
        "alerta_principal": "SEQUIA",
        "produccion_ranking": 3,
    },
    "CUSCO": {
        "lat": -13.5319, "lon": -71.9675,
        "cultivos_principales": ["MAIZ AMILACEO", "PAPA BLANCA", "HABA", "CAFE PERGAMINO", "QUINUA"],
        "cultivos_favorables": {
            "MAIZ AMILACEO":  {"t_min": 10, "t_max": 25, "siembra": [9,10,11], "cosecha": [4,5,6],  "riesgo": "HELADA"},
            "PAPA BLANCA":    {"t_min": 8,  "t_max": 20, "siembra": [10,11],   "cosecha": [3,4,5],  "riesgo": "HELADA"},
            "HABA":           {"t_min": 8,  "t_max": 22, "siembra": [10,11],   "cosecha": [4,5],    "riesgo": "HELADA"},
            "CAFE PERGAMINO": {"t_min": 15, "t_max": 28, "siembra": [10,11],   "cosecha": [5,6,7],  "riesgo": "SEQUIA"},
            "QUINUA":         {"t_min": 5,  "t_max": 20, "siembra": [10,11],   "cosecha": [3,4,5],  "riesgo": "HELADA"},
        },
        "perdidas_diego": "Una de las regiones con mayor riesgo por heladas. En 2024 más de 1 millón de hectáreas en riesgo en zonas altoandinas. Papa y maíz son los cultivos más afectados.",
        "alerta_principal": "HELADA",
        "produccion_ranking": 4,
    },
    "PUNO": {
        "lat": -15.8422, "lon": -70.0199,
        "cultivos_principales": ["PAPA NATIVA", "QUINUA", "HABA", "CAÑIHUA", "OCA"],
        "cultivos_favorables": {
            "PAPA NATIVA": {"t_min": 5,  "t_max": 18, "siembra": [10,11,12], "cosecha": [4,5],   "riesgo": "HELADA"},
            "QUINUA":      {"t_min": 4,  "t_max": 18, "siembra": [10,11],    "cosecha": [5,6],   "riesgo": "HELADA"},
            "HABA":        {"t_min": 6,  "t_max": 18, "siembra": [8,9],      "cosecha": [3,4],   "riesgo": "HELADA"},
            "CAÑIHUA":     {"t_min": 3,  "t_max": 15, "siembra": [10,11],    "cosecha": [4,5],   "riesgo": "HELADA"},
            "OCA":         {"t_min": 5,  "t_max": 18, "siembra": [10,11],    "cosecha": [4,5],   "riesgo": "HELADA"},
        },
        "perdidas_diego": "Región con mayor riesgo de heladas del país. Temperatura mínima es el principal factor de pérdida en zonas altoandinas. Más de 1M ha en riesgo en 2024.",
        "alerta_principal": "HELADA",
        "produccion_ranking": 5,
    },
    "PIURA": {
        "lat": -5.1945, "lon": -80.6328,
        "cultivos_principales": ["MANGO", "LIMON ACIDO", "ARROZ CASCARA", "PLATANO", "MAIZ AMARILLO DURO"],
        "cultivos_favorables": {
            "MANGO":            {"t_min": 18, "t_max": 38, "siembra": [11,12,1], "cosecha": [12,1,2], "riesgo": "NINO_COSTERO"},
            "LIMON ACIDO":      {"t_min": 15, "t_max": 35, "siembra": [3,4],     "cosecha": [10,11],  "riesgo": "EXCESO_LLUVIA"},
            "ARROZ CASCARA":    {"t_min": 20, "t_max": 38, "siembra": [12,1,2],  "cosecha": [5,6,7],  "riesgo": "NINO_COSTERO"},
            "PLATANO":          {"t_min": 18, "t_max": 38, "siembra": [1,2,3],   "cosecha": [8,9,10], "riesgo": "EXCESO_LLUVIA"},
            "MAIZ AMARILLO DURO":{"t_min": 18,"t_max": 38, "siembra": [11,12,1], "cosecha": [4,5,6],  "riesgo": "SEQUIA"},
        },
        "perdidas_diego": "Una de las más vulnerables por irregularidad de lluvias e impacto residual del Niño Costero. Afecta principalmente limón y mango.",
        "alerta_principal": "NINO_COSTERO",
        "produccion_ranking": 6,
    },
    "CAJAMARCA": {
        "lat": -7.1638, "lon": -78.5001,
        "cultivos_principales": ["CAFE PERGAMINO", "PAPA BLANCA", "PLATANO", "PALTO", "MAIZ AMILACEO"],
        "cultivos_favorables": {
            "CAFE PERGAMINO": {"t_min": 15, "t_max": 28, "siembra": [10,11], "cosecha": [5,6,7],  "riesgo": "SEQUIA"},
            "PAPA BLANCA":    {"t_min": 8,  "t_max": 20, "siembra": [7,8],   "cosecha": [11,12],  "riesgo": "HELADA"},
            "PLATANO":        {"t_min": 18, "t_max": 32, "siembra": [1,2,3], "cosecha": [8,9,10], "riesgo": "EXCESO_LLUVIA"},
            "PALTO":          {"t_min": 15, "t_max": 30, "siembra": [1,2,3], "cosecha": [4,5,6],  "riesgo": "SEQUIA"},
            "MAIZ AMILACEO":  {"t_min": 10, "t_max": 25, "siembra": [9,10],  "cosecha": [3,4,5],  "riesgo": "HELADA"},
        },
        "perdidas_diego": "Región de alta producción agrícola con riesgo de heladas en zonas andinas y sequías en temporadas secas.",
        "alerta_principal": "SEQUIA",
        "produccion_ranking": 7,
    },
    "AREQUIPA": {
        "lat": -16.3989, "lon": -71.5350,
        "cultivos_principales": ["PALTO", "PAPA BLANCA", "MAIZ CHALA", "CEBOLLA", "AJO"],
        "cultivos_favorables": {
            "PALTO":     {"t_min": 15, "t_max": 30, "siembra": [1,2,3],  "cosecha": [4,5,6,7],  "riesgo": "ESTRES_TERMICO"},
            "PAPA BLANCA":{"t_min": 8, "t_max": 20, "siembra": [6,7],    "cosecha": [9,10],      "riesgo": "HELADA"},
            "MAIZ CHALA": {"t_min": 15,"t_max": 28, "siembra": [5,6],    "cosecha": [8,9],       "riesgo": "SEQUIA"},
            "CEBOLLA":    {"t_min": 13,"t_max": 28, "siembra": [3,4,5],  "cosecha": [8,9,10],    "riesgo": "SEQUIA"},
            "AJO":        {"t_min": 10,"t_max": 25, "siembra": [4,5,6],  "cosecha": [10,11,12],  "riesgo": "SEQUIA"},
        },
        "perdidas_diego": "Riesgo de estrés térmico para palto y sequías para cultivos andinos.",
        "alerta_principal": "SEQUIA",
        "produccion_ranking": 8,
    },
}

MESES_NOMBRES = {
    1:"enero", 2:"febrero", 3:"marzo", 4:"abril",
    5:"mayo", 6:"junio", 7:"julio", 8:"agosto",
    9:"septiembre", 10:"octubre", 11:"noviembre", 12:"diciembre"
}

ALERTAS_DESC = {
    "HELADA":        "⚠️ Riesgo de helada - temperatura mínima peligrosa para cultivos andinos",
    "SEQUIA":        "⚠️ Riesgo de sequía - déficit hídrico puede reducir rendimiento",
    "ESTRES_TERMICO":"⚠️ Estrés térmico - calor nocturno reduce floración y rendimiento",
    "EXCESO_LLUVIA": "⚠️ Exceso de lluvias - riesgo de pudrición de grano en cosecha",
    "NINO_COSTERO":  "⚠️ Zona de impacto El Niño Costero - irregularidad de lluvias",
}

# ══════════════════════════════════════════════════════════════
# LÓGICA DEL MICROSERVICIO
# ══════════════════════════════════════════════════════════════

def obtener_clima_openmeteo(lat: float, lon: float) -> dict:
    """Consulta clima actual y pronóstico 15 días - Open-Meteo (gratuito, sin API key)"""
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={lat}&longitude={lon}"
        f"&current=temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m"
        f"&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max"
        f"&timezone=America%2FLima&forecast_days=16"
    )
    try:
        resp = req_lib.get(url, timeout=10)
        data = resp.json()
        current = data.get("current", {})
        daily = data.get("daily", {})

        # Temperatura mínima proyectada (próximos 15 días)
        t_min_forecast = min(daily.get("temperature_2m_min", [999]))
        t_max_forecast = max(daily.get("temperature_2m_max", [0]))
        t_prom_forecast = sum(daily.get("temperature_2m_max", [0])) / max(len(daily.get("temperature_2m_max", [1])), 1)
        precip_total = sum(daily.get("precipitation_sum", [0]))

        return {
            "temperatura_actual": current.get("temperature_2m", 0),
            "humedad_actual": current.get("relative_humidity_2m", 0),
            "precipitacion_actual_mm": current.get("precipitation", 0),
            "temperatura_min_15d": round(t_min_forecast, 1),
            "temperatura_max_15d": round(t_max_forecast, 1),
            "temperatura_promedio_15d": round(t_prom_forecast, 1),
            "precipitacion_total_15d_mm": round(precip_total, 1),
            "dias_pronostico": len(daily.get("temperature_2m_max", [])),
            "fuente": "Open-Meteo API (tiempo real)",
            "error": None
        }
    except Exception as e:
        return {
            "temperatura_actual": 20.0,
            "humedad_actual": 65.0,
            "precipitacion_actual_mm": 0.0,
            "temperatura_min_15d": 15.0,
            "temperatura_max_15d": 28.0,
            "temperatura_promedio_15d": 21.0,
            "precipitacion_total_15d_mm": 10.0,
            "dias_pronostico": 15,
            "fuente": "Datos de respaldo (sin conexión)",
            "error": str(e)
        }

def determinar_temporada(mes: int, region: str) -> str:
    """Determina la temporada agrícola según región y mes del año"""
    # Regiones costeras vs sierra/selva tienen temporadas distintas
    regiones_costa = ["LAMBAYEQUE", "LA LIBERTAD", "ICA", "PIURA"]
    regiones_sierra = ["CUSCO", "PUNO", "AREQUIPA", "CAJAMARCA"]

    if region in regiones_costa:
        if mes in [12, 1, 2, 3]:
            return "VERANO COSTEÑO - Temporada de mayor calor y cosecha de frutas"
        elif mes in [4, 5, 6]:
            return "OTOÑO - Inicio de temporada fresca, preparación de suelos"
        elif mes in [7, 8, 9]:
            return "INVIERNO COSTEÑO - Temporada fresca, menor radiación solar"
        else:
            return "PRIMAVERA - Inicio de siembras importantes, clima moderado"
    else:  # Sierra / Selva
        if mes in [11, 12, 1, 2, 3]:
            return "TEMPORADA DE LLUVIAS - Principal período de siembra en sierra"
        elif mes in [4, 5]:
            return "FIN DE LLUVIAS - Inicio de cosechas, clima favorable"
        elif mes in [6, 7, 8, 9]:
            return "TEMPORADA SECA - Heladas nocturnas posibles en zonas altoandinas"
        else:
            return "PRE-LLUVIAS - Preparación para siembra, humedad creciente"

def evaluar_riesgo_climatico(clima: dict, region: str, mes: int) -> dict:
    """Evalúa riesgo climático según datos reales del ing. Diego"""
    alertas = []
    riesgo_nivel = "BAJO"

    t_min = clima["temperatura_min_15d"]
    t_max = clima["temperatura_max_15d"]
    precip = clima["precipitacion_total_15d_mm"]

    # Reglas basadas en análisis de Diego
    if t_min < 0:
        alertas.append("🔴 HELADA SEVERA: Temperatura mínima bajo 0°C. Riesgo crítico para papa, quinua y maíz.")
        riesgo_nivel = "CRÍTICO"
    elif t_min < 4:
        alertas.append("🟠 HELADA MODERADA: Temperatura mínima muy baja. Activar riego por aspersión nocturno.")
        riesgo_nivel = "ALTO" if riesgo_nivel != "CRÍTICO" else riesgo_nivel

    if t_max > 35 and region in ["LAMBAYEQUE", "LA LIBERTAD", "ICA"]:
        alertas.append("🔴 ESTRÉS TÉRMICO: Temperatura máxima >35°C afecta floración de palta, uva y arándanos (-30% a -50% productividad reportado en 2024).")
        riesgo_nivel = "CRÍTICO" if riesgo_nivel not in ["CRÍTICO"] else riesgo_nivel

    if precip > 100 and mes in [6, 7, 8]:
        alertas.append("🟠 EXCESO DE LLUVIAS en época seca: Riesgo de pudrición de grano.")
        riesgo_nivel = "ALTO" if riesgo_nivel == "BAJO" else riesgo_nivel

    if precip < 5 and mes in [1, 2, 3] and region in ["CUSCO", "PUNO", "CAJAMARCA"]:
        alertas.append("🟡 DÉFICIT HÍDRICO: Lluvia muy baja en temporada lluviosa. Verificar fuentes de irrigación.")
        riesgo_nivel = "MODERADO" if riesgo_nivel == "BAJO" else riesgo_nivel

    if region == "PIURA" and mes in [1, 2, 3] and t_max > 33:
        alertas.append("🟠 ALERTA EL NIÑO COSTERO: Condiciones de calor y lluvias irregulares. Alto impacto en limón y mango.")
        riesgo_nivel = "ALTO" if riesgo_nivel == "BAJO" else riesgo_nivel

    if not alertas:
        alertas.append("✅ Condiciones climáticas normales para la región y temporada.")

    return {"nivel": riesgo_nivel, "alertas": alertas}

def recomendar_cultivos(clima: dict, region_data: dict, mes: int) -> list:
    """Recomienda cultivos según clima actual + conocimiento del dataset ENA 2024"""
    recomendaciones = []
    t_min = clima["temperatura_min_15d"]
    t_max = clima["temperatura_max_15d"]

    for cultivo, info in region_data["cultivos_favorables"].items():
        puntuacion = 0
        razon = []

        # ¿La temperatura es compatible?
        if info["t_min"] <= t_min and t_max <= info["t_max"] + 3:
            puntuacion += 40
            razon.append(f"temperatura compatible ({t_min}°C - {t_max}°C)")
        elif t_min < info["t_min"] - 2:
            razon.append(f"⚠️ temperatura mínima muy baja para este cultivo")
        elif t_max > info["t_max"] + 5:
            razon.append(f"⚠️ temperatura máxima demasiado alta")

        # ¿Es época de siembra?
        if mes in info["siembra"]:
            puntuacion += 35
            razon.append("es época de siembra")

        # ¿Es época de cosecha?
        if mes in info["cosecha"]:
            puntuacion += 25
            razon.append("es época de cosecha")

        # Penalizar por riesgo climático activo
        if info["riesgo"] == "HELADA" and t_min < 4:
            puntuacion -= 30
            razon.append("⚠️ riesgo activo de helada")
        if info["riesgo"] == "ESTRES_TERMICO" and t_max > 35:
            puntuacion -= 20
            razon.append("⚠️ estrés térmico activo")

        recomendaciones.append({
            "cultivo": cultivo,
            "puntuacion": puntuacion,
            "razon": ", ".join(razon) if razon else "evaluación climática general",
            "riesgo_principal": info["riesgo"]
        })

    recomendaciones.sort(key=lambda x: x["puntuacion"], reverse=True)
    return recomendaciones[:5]

# ══════════════════════════════════════════════════════════════
# FASTAPI - ENDPOINTS
# ══════════════════════════════════════════════════════════════

app = FastAPI(
    title="🌱 AgroClima Perú - Microservicio Predictivo",
    description=(
        "Microservicio agroclimático para el Perú. "
        "Integra datos de la Encuesta Nacional Agropecuaria 2024 (ENA), "
        "análisis de pérdidas climáticas por región, y pronóstico en tiempo real vía Open-Meteo. "
        "Desarrollado por Equipo 3 - Curso de IA, URP."
    ),
    version="2.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class ConsultaRegion(BaseModel):
    region: str  # Ej: "LAMBAYEQUE", "CUSCO", "ICA"
    provincia: Optional[str] = None  # Ej: "CHICLAYO"
    distrito: Optional[str] = None   # Ej: "PIMENTEL"

@app.get("/")
def root():
    return {
        "servicio": "AgroClima Perú",
        "version": "2.0.0",
        "descripcion": "Predicción agroclimática personalizada por región",
        "regiones_disponibles": list(REGIONES_DATA.keys()),
        "endpoints": {
            "/prediccion/{region}": "GET - Predicción completa por región",
            "/consulta": "POST - Consulta detallada con provincia/distrito",
            "/regiones": "GET - Lista de regiones disponibles con info básica",
            "/clima/{region}": "GET - Solo datos climáticos en tiempo real",
        },
        "fuentes": [
            "ENA 2024 - Encuesta Nacional Agropecuaria (Módulos 1893, 1894, 1895)",
            "Open-Meteo API (clima en tiempo real)",
            "Análisis de pérdidas climáticas 2024-2025"
        ]
    }

@app.get("/regiones")
def listar_regiones():
    resultado = {}
    for region, data in REGIONES_DATA.items():
        resultado[region] = {
            "cultivos_principales": data["cultivos_principales"],
            "alerta_principal": data["alerta_principal"],
            "perdidas_info": data["perdidas_diego"][:100] + "...",
        }
    return resultado

@app.get("/clima/{region}")
def clima_region(region: str):
    region = region.upper()
    if region not in REGIONES_DATA:
        return {"error": f"Región '{region}' no encontrada. Disponibles: {list(REGIONES_DATA.keys())}"}

    data = REGIONES_DATA[region]
    clima = obtener_clima_openmeteo(data["lat"], data["lon"])
    return {
        "region": region,
        "coordenadas": {"lat": data["lat"], "lon": data["lon"]},
        "clima": clima
    }

@app.get("/prediccion/{region}")
def prediccion_por_region(region: str):
    region = region.upper()
    if region not in REGIONES_DATA:
        return {
            "error": f"Región '{region}' no disponible.",
            "regiones_validas": list(REGIONES_DATA.keys())
        }

    region_data = REGIONES_DATA[region]
    mes_actual = datetime.now().month
    mes_nombre = MESES_NOMBRES[mes_actual]

    # 1. Obtener clima real
    clima = obtener_clima_openmeteo(region_data["lat"], region_data["lon"])

    # 2. Determinar temporada
    temporada = determinar_temporada(mes_actual, region)

    # 3. Evaluar riesgo (análisis de Diego)
    riesgo = evaluar_riesgo_climatico(clima, region, mes_actual)

    # 4. Recomendar cultivos (datos ENA 2024 de Benjamín)
    recomendaciones = recomendar_cultivos(clima, region_data, mes_actual)

    # 5. Respuesta estructurada (formato solicitado por ing. Raúl)
    top_cultivo = recomendaciones[0]["cultivo"] if recomendaciones else "N/A"

    return {
        "region": region,
        "fecha_consulta": datetime.now().strftime("%d/%m/%Y %H:%M"),
        "mes_actual": f"{mes_nombre.capitalize()} ({mes_actual})",

        # Resumen ejecutivo (ejemplo del ing. Raúl)
        "resumen": (
            f"Estás en {region.title()} → "
            f"los principales productos de la zona son {', '.join(region_data['cultivos_principales'][:3])} → "
            f"dado que estamos en {mes_nombre} con temperatura de {clima['temperatura_actual']}°C, "
            f"el cultivo más conveniente ahora es {top_cultivo}."
        ),

        "temporada_actual": temporada,

        "clima_actual": {
            "temperatura_c": clima["temperatura_actual"],
            "humedad_pct": clima["humedad_actual"],
            "precipitacion_mm": clima["precipitacion_actual_mm"],
            "fuente": clima["fuente"]
        },

        "pronostico_15_dias": {
            "temperatura_min": clima["temperatura_min_15d"],
            "temperatura_max": clima["temperatura_max_15d"],
            "temperatura_promedio": clima["temperatura_promedio_15d"],
            "precipitacion_total_mm": clima["precipitacion_total_15d_mm"],
        },

        "riesgo_climatico": {
            "nivel": riesgo["nivel"],
            "alertas": riesgo["alertas"],
            "contexto_regional": region_data["perdidas_diego"]
        },

        "recomendaciones_cultivos": [
            {
                "posicion": i+1,
                "cultivo": r["cultivo"],
                "razon": r["razon"],
                "riesgo_principal": r["riesgo_principal"]
            }
            for i, r in enumerate(recomendaciones)
        ],

        "cultivos_en_temporada": {
            "siembra_este_mes": [
                c for c, info in region_data["cultivos_favorables"].items()
                if mes_actual in info["siembra"]
            ],
            "cosecha_este_mes": [
                c for c, info in region_data["cultivos_favorables"].items()
                if mes_actual in info["cosecha"]
            ]
        },

        "fuente_datos": "ENA 2024 - Encuesta Nacional Agropecuaria (INEI/MIDAGRI) + Open-Meteo"
    }

@app.post("/consulta")
def consulta_detallada(consulta: ConsultaRegion):
    """Endpoint POST para consulta con provincia y distrito (Fase 1 - granularidad)"""
    region = consulta.region.upper()

    if region not in REGIONES_DATA:
        return {
            "error": f"Región '{region}' no disponible.",
            "regiones_validas": list(REGIONES_DATA.keys())
        }

    # La predicción base es por región; en Fase 2 se ajustaría por provincia/distrito
    resultado = prediccion_por_region(region)

    # Añadir info de granularidad
    resultado["ubicacion_consultada"] = {
        "region": region,
        "provincia": consulta.provincia or "No especificada",
        "distrito": consulta.distrito or "No especificado",
        "nota": (
            "Predicción basada en región. "
            "Para mayor precisión por provincia/distrito, "
            "integrar con datos georreferenciados del dataset ENA 2024 (LATITUD/LONGITUD disponibles)."
        )
    }

    return resultado

# ══════════════════════════════════════════════════════════════
# LANZAR SERVIDOR EN COLAB
# ══════════════════════════════════════════════════════════════

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

import socket

def puerto_libre(puerto):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', puerto)) != 0

if puerto_libre(8000):
    thread = threading.Thread(target=run_server, daemon=True)
    thread.start()
else:
    print("⚡ Servidor ya estaba corriendo — reutilizando instancia")

import time
time.sleep(2)
print("✅ Microservicio AgroClima Perú iniciado en http://localhost:8000")
print("📖 Documentación: http://localhost:8000/docs")
print("🌍 Regiones disponibles:", list(REGIONES_DATA.keys()))



⚡ Servidor ya estaba corriendo — reutilizando instancia
✅ Microservicio AgroClima Perú iniciado en http://localhost:8000
📖 Documentación: http://localhost:8000/docs
🌍 Regiones disponibles: ['LAMBAYEQUE', 'LA LIBERTAD', 'ICA', 'CUSCO', 'PUNO', 'PIURA', 'CAJAMARCA', 'AREQUIPA']


---
## 🧪 Pruebas del Microservicio
Ejecutar las celdas de prueba **después** de que la celda anterior haya impreso `✅ Microservicio iniciado`.

In [57]:
# ── CELDA 3: PRUEBAS DEL MICROSERVICIO ────────────────────────

import requests
import json

BASE = "http://localhost:8000"

# ── PRUEBA 1: Ver todas las regiones ──────────────
print("=" * 60)
print("PRUEBA 1: Regiones disponibles")
print("=" * 60)
r = requests.get(f"{BASE}/regiones")
for region, info in r.json().items():
    print(f"  {region}: {info['cultivos_principales'][:3]}")

# ── PRUEBA 2: Predicción completa para Lambayeque ──
print()
print("=" * 60)
print("PRUEBA 2: Predicción completa - LAMBAYEQUE")
print("=" * 60)
r = requests.get(f"{BASE}/prediccion/LAMBAYEQUE")
data = r.json()
print(f"\n🌱 RESUMEN: {data['resumen']}")
print(f"\n📅 Temporada: {data['temporada_actual']}")
print(f"\n🌡️  Clima actual: {data['clima_actual']['temperatura_c']}°C, "
      f"Humedad: {data['clima_actual']['humedad_pct']}%")
print(f"\n📊 Pronóstico 15 días: "
      f"Mín {data['pronostico_15_dias']['temperatura_min']}°C / "
      f"Máx {data['pronostico_15_dias']['temperatura_max']}°C / "
      f"Lluvia: {data['pronostico_15_dias']['precipitacion_total_mm']}mm")
print(f"\n⚠️  Riesgo climático: {data['riesgo_climatico']['nivel']}")
for alerta in data['riesgo_climatico']['alertas']:
    print(f"   {alerta}")
print(f"\n🌾 Top cultivos recomendados:")
for rec in data['recomendaciones_cultivos']:
    print(f"   {rec['posicion']}. {rec['cultivo']} - {rec['razon']}")
print(f"\n📅 En siembra este mes: {data['cultivos_en_temporada']['siembra_este_mes']}")
print(f"📅 En cosecha este mes: {data['cultivos_en_temporada']['cosecha_este_mes']}")

# ── PRUEBA 3: Consulta POST con provincia/distrito ──
print()
print("=" * 60)
print("PRUEBA 3: Consulta detallada - CUSCO / Anta / Zurite")
print("=" * 60)
payload = {
    "region": "CUSCO",
    "provincia": "ANTA",
    "distrito": "ZURITE"
}
r = requests.post(f"{BASE}/consulta", json=payload)
data = r.json()
print(f"\n🌱 RESUMEN: {data['resumen']}")
print(f"\n⚠️  Riesgo: {data['riesgo_climatico']['nivel']}")
for alerta in data['riesgo_climatico']['alertas']:
    print(f"   {alerta}")
print(f"\n🌾 Top cultivos:")
for rec in data['recomendaciones_cultivos'][:3]:
    print(f"   {rec['posicion']}. {rec['cultivo']} - {rec['razon']}")
print(f"\n📍 Ubicación consultada: {data['ubicacion_consultada']}")

# ── PRUEBA 4: Solo clima de Puno ──
print()
print("=" * 60)
print("PRUEBA 4: Clima en tiempo real - PUNO")
print("=" * 60)
r = requests.get(f"{BASE}/clima/PUNO")
data = r.json()
clima = data['clima']
print(f"  Temperatura actual: {clima['temperatura_actual']}°C")
print(f"  Humedad: {clima['humedad_actual']}%")
print(f"  Temp mín 15 días: {clima['temperatura_min_15d']}°C")
print(f"  Temp máx 15 días: {clima['temperatura_max_15d']}°C")
print(f"  Lluvia total 15 días: {clima['precipitacion_total_15d_mm']}mm")
print(f"  Fuente: {clima['fuente']}")

print()
print("✅ Todas las pruebas completadas")
print(f"📖 Ver documentación completa en: {BASE}/docs")



PRUEBA 1: Regiones disponibles
  LAMBAYEQUE: ['MANGO', 'PLATANO', 'ARROZ CASCARA']
  LA LIBERTAD: ['PALTO', 'ARANDANO', 'ESPARRAGO']
  ICA: ['VID', 'PALTO', 'ESPARRAGO']
  CUSCO: ['MAIZ AMILACEO', 'PAPA BLANCA', 'HABA']
  PUNO: ['PAPA NATIVA', 'QUINUA', 'HABA']
  PIURA: ['MANGO', 'LIMON ACIDO', 'ARROZ CASCARA']
  CAJAMARCA: ['CAFE PERGAMINO', 'PAPA BLANCA', 'PLATANO']
  AREQUIPA: ['PALTO', 'PAPA BLANCA', 'MAIZ CHALA']

PRUEBA 2: Predicción completa - LAMBAYEQUE

🌱 RESUMEN: Estás en Lambayeque → los principales productos de la zona son MANGO, PLATANO, ARROZ CASCARA → dado que estamos en abril con temperatura de 22.1°C, el cultivo más conveniente ahora es MAIZ AMARILLO DURO.

📅 Temporada: OTOÑO - Inicio de temporada fresca, preparación de suelos

🌡️  Clima actual: 22.1°C, Humedad: 84%

📊 Pronóstico 15 días: Mín 20.3°C / Máx 27.0°C / Lluvia: 0.3mm

⚠️  Riesgo climático: BAJO
   ✅ Condiciones climáticas normales para la región y temporada.

🌾 Top cultivos recomendados:
   1. MAIZ AMARILLO 

---
## 📍 Consulta con Provincia y Distrito
Permite especificar provincia y distrito para predicciones más localizadas.

In [58]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import requests

# 1. Diccionario de datos para los menús (Basado en tu lógica de La Libertad)
# Puedes agregar más regiones y provincias siguiendo esta estructura
DATA_UBICACIONES = {
    'LA LIBERTAD': {
        'TRUJILLO': ['MOCHE', 'LAREDO', 'VICTOR LARCO HERRERA'],
        'ASCOPE': ['CHICAMA', 'PAIJAN']
    },
    'LAMBAYEQUE': {
        'CHICLAYO': ['PIMENTEL', 'REQUE'],
        'FERREÑAFE': ['PUEBLO NUEVO']
    }
}

def consulta_detallada(region, provincia, distrito):
    clear_output(wait=True)
    display(ui) # Mantiene los menús visibles arriba

    try:
        payload = {
            'region': region,
            'provincia': provincia,
            'distrito': distrito
        }

        # Hacemos la petición POST a tu microservicio
        r = requests.post('http://localhost:8000/consulta', json=payload)
        data = r.json()

        # --- DISEÑO DE LA TARJETA DETALLADA (MODO OSCURO) ---
        riesgo = data.get('riesgo_climatico', {})
        nivel = riesgo.get('nivel', 'BAJO')
        accent = "#00d1b2" if nivel == "BAJO" else "#ffdd57" if nivel == "MODERADO" else "#ff3860"

        cultivos_html = ""
        for rec in data.get('recomendaciones_cultivos', [])[:3]:
            cultivos_html += f"""
            <div style="margin-bottom: 8px; padding: 10px; background: #252525; border-radius: 8px; border-left: 3px solid {accent};">
                <span style="color: {accent}; font-weight: bold;">{rec['posicion']}. {rec['cultivo']}</span>
                <div style="color: #aaa; font-size: 11px;">{rec['razon']}</div>
            </div>
            """

        html_output = f"""
        <div style="font-family: 'Segoe UI', sans-serif; background: #121212; padding: 20px; border-radius: 15px; max-width: 500px; color: white; border: 1px solid #333; margin-top: 20px;">
            <div style="border-bottom: 1px solid #333; padding-bottom: 10px; margin-bottom: 15px;">
                <h3 style="margin: 0; color: #4dabf7; font-size: 14px; text-transform: uppercase;">Consulta Detallada</h3>
                <div style="font-size: 18px; font-weight: bold;">{provincia} / {distrito}</div>
            </div>

            <p style="font-size: 13px; color: #eee; line-height: 1.4; background: #1e1e1e; padding: 10px; border-radius: 8px;">
                🌱 {data.get('resumen', 'Cargando...')}
            </p>

            <div style="background: rgba(255,255,255,0.03); border-left: 4px solid {accent}; padding: 12px; border-radius: 5px; margin: 15px 0;">
                <div style="color: {accent}; font-weight: bold; font-size: 12px;">⚠️ RIESGO {nivel}</div>
                <div style="color: #ccc; font-size: 11px;">{riesgo.get('alertas', ['Estable'])[0]}</div>
            </div>

            <div style="font-size: 11px; color: #555; font-weight: bold; margin-bottom: 10px; text-transform: uppercase;">Top Recomendados</div>
            {cultivos_html}

            <div style="margin-top: 15px; font-size: 10px; color: #666; font-style: italic;">
                📌 Nota: {data.get('ubicacion_consultada', {}).get('nota', 'Sin notas adicionales.')}
            </div>
        </div>
        """
        display(HTML(html_output))

    except Exception as e:
        print(f"Error en la consulta: {e}")

# --- LÓGICA DE LOS MENÚS DINÁMICOS ---
w_region = widgets.Dropdown(options=DATA_UBICACIONES.keys(), description='Región:')
w_provincia = widgets.Dropdown(description='Provincia:')
w_distrito = widgets.Dropdown(description='Distrito:')

def actualizar_provincias(*args):
    provincias = DATA_UBICACIONES[w_region.value].keys()
    w_provincia.options = provincias
    actualizar_distritos()

def actualizar_distritos(*args):
    distritos = DATA_UBICACIONES[w_region.value][w_provincia.value]
    w_distrito.options = distritos

w_region.observe(actualizar_provincias, 'value')
w_provincia.observe(actualizar_distritos, 'value')

# Botón para ejecutar la consulta POST
btn = widgets.Button(description="Consultar AgroClima", button_style='info', icon='search')

def clic_consultar(b):
    consulta_detallada(w_region.value, w_provincia.value, w_distrito.value)

btn.on_click(clic_consultar)

# Visualización inicial
ui = widgets.VBox([w_region, w_provincia, w_distrito, btn])
actualizar_provincias() # Carga inicial
display(ui)

In [59]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import requests

# 1. Lista de regiones (asegúrate de que coincidan con tu servidor)
regiones_lista = ['LAMBAYEQUE', 'LA LIBERTAD', 'ICA', 'CUSCO', 'PUNO', 'PIURA', 'CAJAMARCA', 'AREQUIPA']

# 2. Función principal que dibuja todo
def mostrar_dashboard(region):
    # Intentamos obtener los datos del microservicio
    try:
        r = requests.get(f"http://localhost:8000/prediccion/{region}")
        data = r.json()

        clima = data.get('clima_actual', {})
        pronostico = data.get('pronostico_15_dias', {})
        riesgo = data.get('riesgo_climatico', {})
        cultivos_lista = data.get('recomendaciones_cultivos', [])

        # Variables de pronóstico con los nombres exactos de tu código
        t_min = pronostico.get('temperatura_min', '--')
        t_max = pronostico.get('temperatura_max', '--')
        nivel_riesgo = riesgo.get('nivel', 'BAJO')

        # Colores para el Modo Oscuro
        color_map = {
            "BAJO": "#00d1b2",
            "MODERADO": "#ffdd57",
            "ALTO": "#ff3860"
        }
        accent = color_map.get(nivel_riesgo, "#4dabf7")

        # Generar lista de cultivos en HTML
        cultivos_html = ""
        for c in cultivos_lista:
            cultivos_html += f"""
            <div style="margin-bottom: 8px; padding: 10px; background: #252525; border-radius: 8px; border-left: 3px solid {accent};">
                <div style="color: {accent}; font-weight: bold; font-size: 13px;">🌱 {c.get('cultivo')}</div>
                <div style="color: #aaa; font-size: 11px; margin-top: 2px;">{c.get('razon')}</div>
            </div>
            """

        # Estructura de la Tarjeta Modo Oscuro
        html_final = f"""
        <div style="font-family: 'Segoe UI', sans-serif; background: #121212; padding: 20px; border-radius: 15px; max-width: 450px; color: white; border: 1px solid #333; margin-top: 20px;">
            <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 20px;">
                <h2 style="margin: 0; font-size: 22px; color: white;">📍 {region}</h2>
                <span style="background: #333; padding: 4px 12px; border-radius: 20px; font-size: 10px; color: #aaa;">{data.get('temporada_actual', 'OTOÑO')}</span>
            </div>

            <div style="display: flex; gap: 10px; margin-bottom: 20px; text-align: center;">
                <div style="flex: 1; background: #1e1e1e; padding: 12px; border-radius: 10px; border: 1px solid #333;">
                    <div style="font-size: 10px; color: #666; margin-bottom: 5px;">TEMP. ACTUAL</div>
                    <div style="font-size: 20px; font-weight: bold; color: {accent};">{clima.get('temperatura_c', '--')}°C</div>
                </div>
                <div style="flex: 1; background: #1e1e1e; padding: 12px; border-radius: 10px; border: 1px solid #333;">
                    <div style="font-size: 10px; color: #666; margin-bottom: 5px;">HUMEDAD</div>
                    <div style="font-size: 20px; font-weight: bold;">{clima.get('humedad_pct', '--')}%</div>
                </div>
                <div style="flex: 1; background: #1e1e1e; padding: 12px; border-radius: 10px; border: 1px solid #333;">
                    <div style="font-size: 10px; color: #666; margin-bottom: 5px;">RANGO 15D</div>
                    <div style="font-size: 14px; font-weight: bold; margin-top: 5px;">{t_min}° / {t_max}°</div>
                </div>
            </div>

            <div style="background: rgba(255,255,255,0.03); border-left: 4px solid {accent}; padding: 15px; border-radius: 5px; margin-bottom: 20px;">
                <div style="color: {accent}; font-weight: bold; font-size: 12px; text-transform: uppercase;">⚠️ Riesgo {nivel_riesgo}</div>
                <div style="color: #ccc; font-size: 11px; margin-top: 5px; line-height: 1.4;">{riesgo.get('alertas', ['Normal'])[0]}</div>
            </div>

            <div style="border-top: 1px solid #333; padding-top: 15px;">
                <div style="font-size: 11px; color: #555; font-weight: bold; margin-bottom: 12px; text-transform: uppercase;">Recomendaciones</div>
                {cultivos_html}
            </div>
        </div>
        """
        display(HTML(html_final))

    except Exception as e:
        display(HTML(f"<div style='color: #ff3860; padding: 20px;'>❌ Error: No se pudo conectar con el microservicio. Asegúrate de que la celda del servidor esté corriendo.</div>"))

# 3. Lógica de control para evitar el "bloqueo"
def on_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        clear_output(wait=True)
        display(selector)
        mostrar_dashboard(change['new'])

# Crear el widget
selector = widgets.Dropdown(
    options=regiones_lista,
    value=regiones_lista[0],
    description='🔍 Región:',
    style={'description_width': 'initial'}
)

# Configurar la observación
selector.observe(on_change)

# Iniciar la vista
display(selector)
mostrar_dashboard(regiones_lista[0])

Dropdown(description='🔍 Región:', options=('LAMBAYEQUE', 'LA LIBERTAD', 'ICA', 'CUSCO', 'PUNO', 'PIURA', 'CAJA…

In [60]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import requests

# 1. Configuración de datos (Asegúrate de que coincidan con tu Celda 2)
DATA_UBICACIONES = {
    'LA LIBERTAD': {
        'TRUJILLO': ['MOCHE', 'LAREDO', 'VICTOR LARCO HERRERA'],
        'ASCOPE': ['CHICAMA', 'PAIJAN']
    },
    'LAMBAYEQUE': {
        'CHICLAYO': ['PIMENTEL', 'REQUE'],
        'FERREÑAFE': ['PUEBLO NUEVO']
    },
    'ICA': {'ICA': ['SALAS', 'SUBTANJALLA']},
    'CUSCO': {'ANTA': ['ZURITE', 'ANTA']},
    'PUNO': {'PUNO': ['ACORA', 'PLATERIA']},
    'PIURA': {'PIURA': ['CASTILLA', 'CATACAOS']},
    'CAJAMARCA': {'CAJAMARCA': ['ENCAÑADA', 'LOS BAÑOS DEL INCA']},
    'AREQUIPA': {'AREQUIPA': ['YURA', 'YANAHUARA']}
}

# 2. Función para dibujar la tarjeta (sirve para ambos tipos de consulta)
def dibujar_tarjeta(data, es_detallado=False):
    clima = data.get('clima_actual', {})
    pronostico = data.get('pronostico_15_dias', {})
    riesgo = data.get('riesgo_climatico', {})
    nivel = riesgo.get('nivel', 'BAJO')
    accent = "#00d1b2" if nivel == "BAJO" else "#ffdd57" if nivel == "MODERADO" else "#ff3860"

    t_min = pronostico.get('temperatura_min', '--')
    t_max = pronostico.get('temperatura_max', '--')

    cultivos_html = ""
    for c in data.get('recomendaciones_cultivos', [])[:3]:
        cultivos_html += f"""
        <div style="margin-bottom: 8px; padding: 10px; background: #252525; border-radius: 8px; border-left: 3px solid {accent};">
            <div style="color: {accent}; font-weight: bold; font-size: 13px;">🌱 {c.get('cultivo')}</div>
            <div style="color: #aaa; font-size: 11px;">{c.get('razon')}</div>
        </div>
        """

    header_title = f"{data.get('region')}"
    if es_detallado:
        ub = data.get('ubicacion_consultada', {})
        header_title = f"{ub.get('provincia')} / {ub.get('distrito')}"

    html = f"""
    <div style="font-family: 'Segoe UI', sans-serif; background: #121212; padding: 20px; border-radius: 15px; max-width: 450px; color: white; border: 1px solid #333; margin-top: 20px;">
        <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 20px;">
            <h2 style="margin: 0; font-size: 20px;">📍 {header_title}</h2>
            <span style="background: #333; padding: 4px 10px; border-radius: 20px; font-size: 10px; color: #888;">{data.get('temporada_actual', 'OTOÑO')}</span>
        </div>

        <div style="display: flex; gap: 10px; margin-bottom: 20px; text-align: center;">
            <div style="flex: 1; background: #1e1e1e; padding: 10px; border-radius: 10px; border: 1px solid #333;">
                <div style="font-size: 9px; color: #666;">TEMP. ACTUAL</div>
                <div style="font-size: 18px; font-weight: bold; color: #4dabf7;">{clima.get('temperatura_c', '--')}°C</div>
            </div>
            <div style="flex: 1; background: #1e1e1e; padding: 10px; border-radius: 10px; border: 1px solid #333;">
                <div style="font-size: 9px; color: #666;">HUMEDAD</div>
                <div style="font-size: 18px; font-weight: bold;">{clima.get('humedad_pct', '--')}%</div>
            </div>
            <div style="flex: 1; background: #1e1e1e; padding: 10px; border-radius: 10px; border: 1px solid #333;">
                <div style="font-size: 9px; color: #666;">PRONÓSTICO</div>
                <div style="font-size: 13px; font-weight: bold; margin-top: 4px;">{t_min}° / {t_max}°</div>
            </div>
        </div>

        <div style="background: rgba(255,255,255,0.03); border-left: 4px solid {accent}; padding: 12px; border-radius: 5px; margin-bottom: 15px;">
            <div style="color: {accent}; font-weight: bold; font-size: 12px;">⚠️ RIESGO {nivel}</div>
            <div style="color: #ddd; font-size: 11px;">{riesgo.get('alertas', ['Estable'])[0]}</div>
        </div>

        <div style="border-top: 1px solid #333; padding-top: 15px;">
            <div style="font-size: 11px; color: #555; font-weight: bold; margin-bottom: 10px; text-transform: uppercase;">Top Recomendaciones</div>
            {cultivos_html}
        </div>
    </div>
    """
    display(HTML(html))

# 3. Lógica de los widgets
w_region = widgets.Dropdown(options=DATA_UBICACIONES.keys(), description='Región:')
w_provincia = widgets.Dropdown(description='Provincia:')
w_distrito = widgets.Dropdown(description='Distrito:')
btn_consulta = widgets.Button(description="Consulta Detallada (POST)", button_style='info', layout={'width': 'max-content'})

def actualizar_interfaz(region):
    # Actualiza provincias basándose en la región
    provincias = list(DATA_UBICACIONES[region].keys())
    w_provincia.options = provincias
    actualizar_distritos(provincias[0])

    # Realiza consulta GET automática para la región (Dashboard General)
    try:
        r = requests.get(f"http://localhost:8000/prediccion/{region}")
        dibujar_tarjeta(r.json())
    except:
        print("Error conectando al servidor...")

def actualizar_distritos(provincia):
    distritos = DATA_UBICACIONES[w_region.value][provincia]
    w_distrito.options = distritos

def manejar_cambio_region(change):
    clear_output(wait=True)
    display(interfaz_completa)
    actualizar_interfaz(change['new'])

def manejar_clic_post(b):
    try:
        payload = {'region': w_region.value, 'provincia': w_provincia.value, 'distrito': w_distrito.value}
        r = requests.post('http://localhost:8000/consulta', json=payload)
        clear_output(wait=True)
        display(interfaz_completa)
        dibujar_tarjeta(r.json(), es_detallado=True)
    except:
        print("Error en consulta POST")

# Eventos
w_region.observe(manejar_cambio_region, names='value')
w_provincia.observe(lambda c: actualizar_distritos(c['new']), names='value')
btn_consulta.on_click(manejar_clic_post)

# Layout final
controles = widgets.VBox([
    widgets.HTML("<b style='color:#4dabf7'>🛠️ PANEL DE CONTROL AGROCLIMA</b>"),
    w_region, w_provincia, w_distrito, btn_consulta
])
interfaz_completa = widgets.Box([controles], layout={'padding': '15px', 'border': '1px solid #333', 'width': 'max-content', 'border_radius': '10px'})

# Inicio
display(interfaz_completa)
actualizar_interfaz(w_region.value)

Box(children=(VBox(children=(HTML(value="<b style='color:#4dabf7'>🛠️ PANEL DE CONTROL AGROCLIMA</b>"), Dropdow…